# Taasesinevad närvivõrgud

Eelmises moodulis käsitlesime teksti rikkalikke semantilisi esitlusi. Kasutatud arhitektuur tabab sõnade lauses üldistatud tähendust, kuid see ei võta arvesse sõnade **järjekorda**, kuna kokkuvõttev operatsioon, mis järgneb manustamisele, eemaldab selle info algsest tekstist. Kuna need mudelid ei suuda esitada sõnade järjekorda, ei saa nad lahendada keerukamaid või ambivalentsemaid ülesandeid nagu teksti genereerimine või küsimustele vastamine.

Tekstijada tähenduse tabamiseks kasutame närvivõrgu arhitektuuri, mida nimetatakse **taasesinevaks närvivõrguks** ehk RNN-ks. RNN-i kasutades saadame oma lause läbi võrgu ühe märgi kaupa ning võrk toodab mingi **oleku**, mille seejärel edastame võrku koos järgmise märgiga.

![Pilt, mis näitab taasesineva närvivõrgu näite genereerimist.](../../../../../translated_images/et/rnn.27f5c29c53d727b5.webp)

Antud sisendsõnumite jadale $X_0,\dots,X_n$ loob RNN ühe närvivõrgublokkide jada ning koolitab selle jadana lõpust lõpuni kasutades tagasipropageerimist. Iga võrguplokk võtab sisendiks paari $(X_i,S_i)$ ja annab tulemuseks $S_{i+1}$. Lõplik olek $S_n$ või väljund $Y_n$ läheb lineaarse klassifikaatori sisendiks tulemuse saamiseks. Kõik plokid kasutavad samu kaalusid ning neid koolitatakse lõpuni kasutades üht tagasipropageerimise läbimist.

> Ülaloleval joonisel on taasesinev närvivõrk näidatud lahtirullituna (vasakul) ja kompaktsema taasesineva esituse kujul (paremal). Oluline on mõista, et kõik RNN-i seadmed kasutavad samu **jagamisi kaalusid**.

Kuna olekuvektorid $S_0,\dots,S_n$ edastatakse läbi võrgu, suudab RNN õppida sõnade järjestusevahelisi seoseid. Näiteks, kui sõna *not* ilmub kuskil jadades, võib võrk õppida selle abil olekuvektoris teatud elemente eitama.

Iga RNN-i seadme sees on kaks kaalumaatriksit: $W_H$ ja $W_I$ ning nihe $b$. Iga RNN sammu juures, antud sisend $X_i$ ja sisendolek $S_i$, arvutatakse väljundolek kui $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, kus $f$ on aktiveerimisfunktsioon (sagedasti $\tanh$).

> Teksti genereerimise probleemi puhul (mida käsitleme järgmises osas) või masintõlkes soovime samuti saada väljundväärtuse iga RNN sammu juures. Sel juhul on veel üks maatriks $W_O$ ning väljund arvutatakse kui $Y_i=f(W_O\times S_i+b_O)$.

Vaatame, kuidas taasesinevad närvivõrgud aitavad uudiste andmekogumit klassifitseerida.

> Liivakasti keskkonnas tuleb jooksutada järgnev lahter, et veenduda vajaliku raamatukogu paigaldamises ning andmete eeltoomises. Kui töötate kohalikus keskkonnas, võite selle lahtri vahele jätta.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Suuremate mudelite treenimisel võib GPU mälumahutuse jaotamine saada probleemiks. Võib-olla peame ka katsetama erinevate minibatchi suurustega, nii et andmed mahuksid meie GPU mällu, kuid treenimine oleks siiski piisavalt kiire. Kui jooksutate seda koodi oma GPU masinal, võite katsetada minibatchi suuruse reguleerimist treeningu kiirendamiseks.

> **Märkus**: Teatud NVidia draiverite versioonid ei vabasta mälu pärast mudeli treenimist. Me jooksutame selles märkmikus mitmeid näiteid ja see võib mõnes seades põhjustada mälumahtu ammendumise, eriti kui teete oma katseid sama märkmiku raames. Kui mudeli treenimise alustamisel tekib kummalisi vigu, võite proovida märkmiku tuuma taaskäivitada.


In [3]:
batch_size = 16
embed_size = 64

## Lihtne RNN klassifikaator

Lihtsa RNN-i puhul on iga korduv üksus lihtne lineaarne võrk, mis võtab sisendvektori ja olekuvektori ning toodab uue olekuvektori. Keras saab seda esindada kihiga `SimpleRNN`.

Kuigi me võiksime RNN kihile otse anda one-hot kodeeritud tokeneid, ei ole see hea mõte nende kõrge dimensioonide arvu tõttu. Seetõttu kasutame sõnade vektorite dimensiooni vähendamiseks embed-kihi, millele järgneb RNN kiht ja lõpuks `Dense` klassifikaator.

> **Märkus**: Juhtudel, kui dimensioonide arv ei ole nii kõrge, näiteks tähepoolse tokenizeerimise kasutamisel, võib olla mõistlik anda one-hot kodeeritud tokenid otse RNN raku sisendiks.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Märkus:** Siin kasutame lihtsuse huvides treenimata manustamiskihi, kuid paremate tulemuste saavutamiseks võime kasutada eelnevalt treenitud manustamiskihte Word2Vec'iga, nagu on kirjeldatud eelnevas üksuses. Hea harjutus oleks kohandada see kood nii, et see töötaks eelnevalt treenitud manustamistega.

Treenime nüüd oma RNN-i. RNN-e on üldiselt üsna keeruline treenida, sest kui RNN-i rakud lahti rullitakse järjestuse pikkuses, on tagasipropagatsioonis osalevate kihtide arv üsna suur. Seetõttu peame valima väiksema õppemäära ja treenima võrku suurema andmestiku peal, et saada häid tulemusi. See võib võtta üsna kaua aega, seega eelistatakse kasutada GPU-d.

Kiirendamiseks treenime RNN mudelit ainult uudiste pealkirjade peal, jättes kirjelduse välja. Võid proovida treenida ka kirjeldusega ja vaadata, kas mudel treenib siis edukalt.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Märkus:** täpsus võib siin tõenäoliselt olla madalam, kuna me treenime ainult uudiste pealkirjadega.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Nüüd, kus me kasutame maskimist, saame mudelit koolitada kogu pealkirjade ja kirjelduste andmestikul.

> **Märkus**: Kas olete märganud, et oleme kasutanud vektoriseerijat, mis on treenitud uudiste pealkirjade peal, mitte kogu artikli sisu peal? Võimalik, et see võib põhjustada mõningate tokendeid eiramist, seega on parem vektoriseerijat uuesti treenida. Siiski võib sellel olla ainult väga väike mõju, nii et me jääme varasema eelnevalt treenitud vektoriseerija juurde lihtsuse huvides.


## LSTM: Pikajaline lühimälu

Üks peamisi RNN-ide probleeme on **kaduvad gradientid**. RNN-id võivad olla üsna pikad ja neil võib olla raske gradientide levitamist tagasi võrgu esimese kihini tagasiside kaudu. Kui see juhtub, ei suuda võrk õppida kaugete märkide vahelisi seoseid. Üks viis selle probleemi vältimiseks on kasutada **eksplitseeritud oleku haldamist** läbi **väravate**. Kaks kõige tavalisemat arhitektuuri, mis väravaid kasutavad, on **pikajaline lühimälu** (LSTM) ja **väravatud releeüksus** (GRU). Siin käsitleme LSTM-i.

![Pilt, mis näitab pikajalise lühimälu rakku](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM-võrk on organiseeritud sarnaselt RNN-ile, kuid kihist kihi läbib kaks olekut: tegelik olek $c$ ja peidetud vektor $h$. Igas üksuses kombineeritakse varasem peidetud vektor $h_{t-1}$ sisendiga $x_t$ ning nad koos kontrollivad, mis toimub oleku $c_t$ ja väljundiga $h_{t}$ läbi **väravate**. Igal väraval on sigmoid-aktiveerimine (väärtus vahemikus $[0,1]$), mida võib mõelda bitipõhise maskina, kui see korrutatakse olekuvektoriga. LSTM-il on järgmised väravad (vasakult paremale pildil eespool):
* **unustamise värav**, mis määrab, milliseid olekuvektori $c_{t-1}$ komponente tuleb unustada ja milliseid edasi anda.
* **sisendi värav**, mis määrab, kui palju infot sisendvektorist ja eelmisest peidetud vektorist tuleks olekuvektorisse lisada.
* **väljundi värav**, mis võtab uue olekuvektori ja otsustab, milliseid selle komponente kasutatakse uue peidetud vektori $h_t$ loomiseks.

Oleku $c$ komponendid võib mõelda lipuna, mida saab sisse ja välja lülitada. Näiteks, kui järjestuses kohtame nime *Alice*, oletame, et see viitab naisele, ja tõstame lipu olekus, mis ütleb, et lauses on naissoost nimisõna. Kui kohtame järgmisi sõnu *and Tom*, tõstame lipu, mis ütleb, et tegemist on mitmuse vormis nimisõnaga. Seeläbi saame olekut manipuleerides hoida lause grammatiliste omaduste jälgimist.

> **Märkus**: Siin on suurepärane allikas LSTM-ide toimimise mõistmiseks: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) autoriks Christopher Olah.

Kuigi LSTM-raku sisemine struktuur võib tunduda keeruline, peidab Keras selle rakenduse kihis `LSTM`, seega on vaja ülaltoodud näites asendada ainult korduv kiht:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Pange tähele**, et LSTM-ide koolitamine on samuti üsna aeglane ning esialgu ei pruugi täpsuses palju paranemist näha. Hea täpsuse saavutamiseks võib olla vajalik jätkata koolitamist mõnda aega.


## Kahepoolsed ja mitmekihilised RNNid

Meie seni toodud näidetes töötavad korduvad võrgud järjestuse algusest kuni lõpuni. See tundub loomulik, sest järgib sama suunda, milles me loeme või kuulates kõnet. Kuid juhtudel, kus on vaja sisendjärjestusele juhuslikku ligipääsu, on mõistlikum lasta korduvatel arvutustel toimuda mõlemas suunas. RNN-id, mis võimaldavad arvutusi mõlemas suunas, nimetatakse **kahepoolsed** RNN-id ning neid saab luua, mähkides korduva kihi erilise `Bidirectonal` kihiga.

> **Märkus**: `Bidirectional` kiht teeb selle sees olevast kihist kaks koopiat ning seab ühe koopia `go_backwards` omaduseks `True`, mis paneb selle liikuma järjestuse vastassuunas.

Korduvad võrgud, ühesuunalised või kahepoolsed, püüavad järjestuses kinni mustreid ning salvestavad need olekuvektoritesse või väljastavad neid väljundina. Nagu konvolutsioonvõrkude puhul, saame luua teise korduva kihi esimese järel, et tabada kõrgema tasandi mustreid, mis põhinevad esmalt vastavast kihist eraldatud madalama tasandi mustritel. See viib meid mõisteni **mitmekihiline RNN**, mis koosneb kahest või enamast korduvast võrgust, kus eelmise kihi väljund läheb järgmise kihi sisendiks.

![Pilt mitmekihilisest pikk-mäluga RNN-ist](../../../../../translated_images/et/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Pilt pärit [selles imetlusväärses postituses](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3), autor Fernando López.*

Keras teeb nende võrkude loomise lihtsaks, sest vajad ainult mudelile lisada rohkem korduvaid kihte. Kõikide kihtide puhul peale viimase tuleb määrata `return_sequences=True` parameeter, sest on vaja, et kiht tagastaks kõik vahepealsed olekud, mitte ainult korduva arvutuse lõppoleku.

Loome nüüd kaks-kihilise kahepoolselt ühendatud LSTM-i meie klassifitseerimisülesande jaoks.

> **Märkus** see kood võtab jälle üsna kaua aega, et täita, kuid annab seni kõrgeima täpsuse. Nii et ehk tasub oodata ja tulemust vaadata.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN-id teiste ülesannete jaoks

Seni oleme keskendunud RNN-ide kasutamisele tekstijadade klassifitseerimiseks. Kuid nad suudavad toime tulla ka paljude teiste ülesannetega, nagu teksti genereerimine ja masintõlge &mdash; nende ülesannetega käsitleme järgmisel üksusel.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
